In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

os.listdir(path)

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

file_path = "/asset/imdb_2000_sample.csv"
df = pd.read_csv(file_path)
df.head()

In [ ]:
df.info()

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

**Data Preprocessing**

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [ ]:
import re

def clean_text(text):
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
def preprocess_text(text):
  doc = nlp(text.lower()) # lowercase

  tokens = [
      token.lemma_ #lemmatization
      for token in doc
      if not token.is_punct # remove punctuation
      and not token.is_stop # remove stopwords
      and not token.is_digit # remove numbers
  ]

  return " ".join(tokens)

In [ ]:
def preprocess_batch(texts):
    cleaned = []

    texts = [clean_text(text.lower()) for text in texts]

    for doc in nlp.pipe(texts, batch_size=100):
        tokens = [
            token.lemma_
            for token in doc
            if not token.is_punct
            and not token.is_stop
            and not token.is_digit
            and token.is_alpha
        ]
        cleaned.append(" ".join(tokens))

    return cleaned

In [ ]:
df["cleaned_review"] = preprocess_batch(df["review"])

In [ ]:
df

In [ ]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

**Convert Text into Number**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,      # ignore rare words
    max_df=0.8,    # ignore too common words
    ngram_range=(1,2)
)

In [ ]:
X = vectorizer.fit_transform(df["cleaned_review"])

In [ ]:
y = df["label"]

In [ ]:
print(X.shape)

In [ ]:
print(vectorizer.get_feature_names_out()[:20])

In [ ]:
import pandas as pd

X_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
X_df.head()

**Train Model**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

In [ ]:
# SVM

from sklearn.svm import LinearSVC

svm_model = LinearSVC()

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

In [ ]:
import pickle

pickle.dump(lr_model, open("sentiment_model_lr.pkl", "wb"))
pickle.dump(nb_model, open("sentiment_model_nb.pkl", "wb"))
pickle.dump(svm_model, open("sentiment_model_svm.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf.pkl", "wb"))

In [ ]:
review = ["This movie was amazing!"]

In [ ]:
review_vector = vectorizer.transform(review)

In [ ]:
prediction = svm_model.predict(review_vector)
print(prediction)